# RK1828 Qwen3-Reranker 文档重排序 Demo


[Qwen3-Reranker-0.6B](https://github.com/QwenLM/Qwen3-Embedding)：
给（query, document）对打 0~1 的相关性分数，用于 RAG 精排。GRQ W4A16 量化，
实测单对 prefill 132ms；相关 0.9985 vs 无关 0.0000。

| 步骤 | 内容 |
|---|---|
| 1 | 下载模型（可选 HF / HF-Mirror 源） |
| 2 | 解压预编译 demo |
| 3 | 输入 query 和候选文档 |
| 4 | 打分排序 |

## 0. 环境检查

先确认 RKNN3 运行库（`librknn3_api.so`）和 RK1828 设备都正常。
如失败，请先安装驱动/运行时（`rknn3-rk182x` DEB 包），并用 `sudo rknn-smi info` 确认能看到 RK1828。

In [ ]:
from rknn3 import get_lib, find_devices

_ = get_lib()  # 加载 /usr/lib/librknn3_api.so；失败说明 Runtime 未安装

devices = find_devices()
if not devices:
    raise RuntimeError("未发现 RK1828 设备！请检查: sudo rknn-smi info")
for dev_id, dev_type in devices:
    print(f"发现设备: id={dev_id}  接口={dev_type}")

## 1. 下载模型

本 demo 的模型文件托管在 Hugging Face（`qhddz` 账号）。
**先用下拉框选择下载源**，再运行下一个单元格开始下载：

- **HF-Mirror 国内镜像**：`hf-mirror.com`，国内直连，推荐
- **Hugging Face 官方**：`huggingface.co`，海外或已有代理时用

> 已存在（且非空）的文件会**自动跳过**，重复运行不会重新下载。

In [ ]:
from nb_download import source_picker

# (HF仓库名, 仓库内路径, 本地路径)
FILES = [
    ("qwen3-reranker-0.6b-rknn3-rk1828", "demo_x86_64.tar.gz", "demos/qwen3_reranker/demo_x86_64.tar.gz"),
    ("qwen3-reranker-0.6b-rknn3-rk1828", "model/Qwen3-Reranker-0.6B.embed.bin", "demos/qwen3_reranker/model/Qwen3-Reranker-0.6B.embed.bin"),
    ("qwen3-reranker-0.6b-rknn3-rk1828", "model/Qwen3-Reranker-0.6B.rknn", "demos/qwen3_reranker/model/Qwen3-Reranker-0.6B.rknn"),
    ("qwen3-reranker-0.6b-rknn3-rk1828", "model/Qwen3-Reranker-0.6B.tokenizer.gguf", "demos/qwen3_reranker/model/Qwen3-Reranker-0.6B.tokenizer.gguf"),
    ("qwen3-reranker-0.6b-rknn3-rk1828", "model/Qwen3-Reranker-0.6B.weight", "demos/qwen3_reranker/model/Qwen3-Reranker-0.6B.weight"),
]

w_source = source_picker()

In [ ]:
from nb_download import ensure_files

ensure_files(FILES, w_source.value)

## 2. 解压预编译 demo

`demo_x86_64.tar.gz` 里是官方流程编译好的 x86_64 C++ demo 和运行时库（`lib/`）。
HF 不允许直接托管可执行文件，所以打包成 tar.gz，解压一次即可（重复运行自动跳过）。

> **为什么这个 demo 用 C++ 而不是纯 Python？**
> 01-05（分类/检测/FastSAM/pose/分割）是单个模型一次前向，后处理简单，
> 所以用 Python（`rknn3.py`）逐步演示。而本 demo 的流水线复杂得多——
> 多模型串联、tokenizer、KV cache、滑窗/流式状态等，前后处理和编排逻辑
> 有几百上千行，重写 Python 版成本高且容易和官方实现不一致。
> 因此直接运行官方 `rknn3-model-zoo` 编译的 C++ demo，
> 源码见 [github.com/airockchip/rknn3-model-zoo](https://github.com/airockchip/rknn3-model-zoo)
> 的 `examples/` 目录。

In [ ]:
from nb_download import extract_tarball

extract_tarball("demos/qwen3_reranker/demo_x86_64.tar.gz", "demos/qwen3_reranker")

In [ ]:
from nb_demo import run_demo, snapshot, new_files, show_images, show_new_images, show_audio

## 3. 输入 query 和候选文档（每行一篇）

In [ ]:
import ipywidgets as widgets
from IPython.display import display

w_query = widgets.Text(value="What are gravitational waves?",
                       description="Query:", layout=widgets.Layout(width="75%"))
w_docs = widgets.Textarea(
    value="Gravitational waves are ripples in spacetime caused by accelerating masses.\n"
          "The best chocolate chip cookie recipe uses brown butter.\n"
          "LIGO first detected gravitational waves in 2015, confirming Einstein's prediction.\n"
          "Python is a popular programming language for data science.",
    description="文档(每行一篇):", layout=widgets.Layout(width="75%", height="120px"))
display(w_query, w_docs)
print("选好后继续往下运行 ▼")

## 4. 打分排序

In [ ]:
import re

import numpy as np

DEMO = "demos/qwen3_reranker"
docs = [d.strip() for d in w_docs.value.splitlines() if d.strip()]
scores = []
for d in docs:
    p = run_demo(["./rknn_qwen3_reranker_demo",
                  "model/Qwen3-Reranker-0.6B.rknn", "model/Qwen3-Reranker-0.6B.weight",
                  "model/Qwen3-Reranker-0.6B.tokenizer.gguf",
                  "model/Qwen3-Reranker-0.6B.embed.bin",
                  "0xff", w_query.value, d], DEMO)
    nums = re.findall(r"(?:score|Score|relevance)[:\s]+([0-9]*\.?[0-9]+)", p.stdout)
    if not nums:  # 兜底：取输出中最后一个 0~1 的小数
        nums = re.findall(r"(?<![\d.])(0?\.[0-9]+|1\.0+)(?![\d.])", p.stdout)
    scores.append(float(nums[-1]))
    print(f"  -> {scores[-1]:.4f}")

order = np.argsort(scores)[::-1]
print("\n按相关性排序:")
for i in order:
    print(f"  {scores[i]:.4f}  {docs[i][:60]}")

In [ ]:
import matplotlib.pyplot as plt

labels = [d[:28] for d in docs]
order = np.argsort(scores)
plt.figure(figsize=(8, 3.5))
plt.barh([labels[i] for i in order], [scores[i] for i in order], color="#3b82f6")
plt.xlim(0, 1)
plt.title(f"query: {w_query.value}")
plt.tight_layout()
plt.show()

## 小结

- Reranker 是 RAG 的"精排"：先用 Embedding 粗排出 top-K，再用它逐个精排提准。
- 相关与不相关的分数差非常极端（0.99 vs 0.00），说明模型区分度很好。